# Notebook 22 — Jockey and Trainer Identity

## Bounded question

Can raw `jockey` and `trainer` labels be mapped safely to stable source-internal and, where evidence permits, real-world participant identities without merging different people or splitting the same person unnecessarily?

## Purpose

This notebook investigates the source semantics, physical behaviour and identity risks of the runner-level `jockey` and `trainer` fields.

It is an identity study, not a participant-performance ranking exercise. Raw-label aggregates remain source-label analysis unless and until an identity mapping is supported by evidence.

The investigation will begin with source-wide profiling. It will not assume that punctuation removal, case folding, initial expansion, title removal, whitespace normalisation or fuzzy matching is safe.

The study will distinguish between:

* immutable raw source labels;
* exploratory candidate-match text;
* source-internal participant occurrences;
* governed equivalence decisions;
* governed split decisions;
* unresolved relationships;
* authority- or evidence-backed real-world identities.

No broad name normalisation or cross-role merge is authorised in advance.

## Source and grain

* **Source database:** `data/raw/form_2015-present/form_2015-present/raceform.db`
* **Source table:** `data`
* **Governed data-row predicate:** `rowid <> 1`
* **Fields under investigation:** `jockey`, `trainer`
* **Declared source type:** to be confirmed from SQLite
* **Provisional grain:** runner-level source assertions about race participants
* **Raw preservation:** required
* **Current analytical status:** identity semantics pending
* **Notebook 20 relationship:** existing blank supplementation remains governed and must not be silently replaced or reinterpreted here
* **Notebook 19 relationship:** the pending horse/pedigree authority gate is separate and must not be modified by this notebook

## Scope

The investigation will test:

* SQL null, empty-string, whitespace-only and placeholder behaviour;
* populated and distinct raw-label counts;
* punctuation, spacing, case, diacritic, bracket, title and suffix conventions;
* initials versus expanded names;
* spelling variants and probable source defects;
* same-name collision risk;
* jurisdiction and active-period boundaries;
* one person appearing under several labels;
* one label potentially referring to several people;
* jockey and trainer role distinctions;
* race-level and runner-level occurrence consistency;
* safe source-internal identity keys;
* occurrence splitting where required;
* evidence thresholds for equivalence, correction, splitting and preservation;
* cases that must remain unresolved.

## Closure obligations

If the evidence supports governed identity mappings, closure will require persisted and reloaded outputs, reusable implementation, focused tests, an independent source-wide validator, integration documentation, an explicit manual-verification decision, a reader-facing Minto report, lessons learned, audit and project-status updates, and recorded local validation.

The notebook will proceed one evidence-led stage at a time. Stable implementation and permanent reference outputs will be created only after the observed source behaviour justifies them.


## 1. Establish the source population and physical field behaviour

The first stage confirms the governed runner population, the declared SQLite storage for both fields, and the basic physical behaviour of each raw label.

This stage measures:

* governed runner rows;
* SQL nulls;
* empty strings;
* whitespace-only values;
* populated rows;
* distinct populated raw labels;
* leading and trailing whitespace;
* minimum and maximum populated lengths.

These checks precede any identity interpretation. No label will be trimmed, case-folded, de-punctuated, tokenised or fuzzy-matched in this stage.


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

DATA_ROW_PREDICATE = "rowid <> 1"
PARTICIPANT_FIELDS = ("jockey", "trainer")

assert SOURCE_DB_PATH.exists(), f"Source database not found: {SOURCE_DB_PATH}"

connection = sqlite3.connect(f"file:{SOURCE_DB_PATH}?mode=ro", uri=True)

schema = pd.read_sql_query("PRAGMA table_info(data)", connection)
participant_schema = schema.loc[schema["name"].isin(PARTICIPANT_FIELDS)].copy()

assert set(participant_schema["name"]) == set(PARTICIPANT_FIELDS), (
    "Expected jockey and trainer fields were not both present in the source schema."
)

profile_rows = []

for field_name in PARTICIPANT_FIELDS:
    field_profile = pd.read_sql_query(
        f'''
        SELECT
            '{field_name}' AS field_name,
            COUNT(*) AS governed_runner_rows,
            SUM({field_name} IS NULL) AS null_rows,
            SUM({field_name} = '') AS empty_string_rows,
            SUM(
                {field_name} IS NOT NULL
                AND {field_name} <> ''
                AND TRIM({field_name}) = ''
            ) AS whitespace_only_rows,
            SUM(
                {field_name} IS NOT NULL
                AND TRIM({field_name}) <> ''
            ) AS populated_rows,
            COUNT(
                DISTINCT CASE
                    WHEN {field_name} IS NOT NULL
                         AND TRIM({field_name}) <> ''
                    THEN {field_name}
                END
            ) AS distinct_populated_labels,
            SUM(
                {field_name} IS NOT NULL
                AND {field_name} <> LTRIM({field_name})
            ) AS leading_whitespace_rows,
            SUM(
                {field_name} IS NOT NULL
                AND {field_name} <> RTRIM({field_name})
            ) AS trailing_whitespace_rows,
            MIN(
                CASE
                    WHEN {field_name} IS NOT NULL
                         AND TRIM({field_name}) <> ''
                    THEN LENGTH({field_name})
                END
            ) AS minimum_populated_length,
            MAX(
                CASE
                    WHEN {field_name} IS NOT NULL
                         AND TRIM({field_name}) <> ''
                    THEN LENGTH({field_name})
                END
            ) AS maximum_populated_length
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        ''',
        connection,
    )
    profile_rows.append(field_profile)

participant_physical_profile = pd.concat(profile_rows, ignore_index=True)

display(participant_schema)
display(participant_physical_profile)


### Output review checkpoint

Do not proceed to candidate normalisation from assumptions.

After running the profiling cell, record what the outputs establish about:

* the source missing-value convention for each role;
* whether either field contains physical whitespace anomalies;
* the scale of the distinct-label problem;
* whether unusually short or long labels need direct inspection;
* whether jockey and trainer require separate downstream rules.

The next analytical cell must follow from the observed output.
